# 🚀 Amazing-QR: 9 Adımlı Profesyonel İş Akışı v1.2

Bu notebook, **Amazing-QR** projesini 9 adımda tam kontrollü ve profesyonel bir şekilde çalıştırmak için tasarlanmıştır.

---

## 🛠️ 1. Kurulum ve Hazırlık
Sistemi hazırlar ve gerekli kütüphaneleri yükler.

In [ ]:
import os
import shutil
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

# 1. Projeyi GitHub'dan çek (Veya güncelle)
repo_name = "amazing-qr"
if not os.path.exists(repo_name):
    print("🚀 Proje indiriliyor...")
    !git clone https://github.com/orioninsist/amazing-qr.git
    %cd {repo_name}
else:
    %cd {repo_name}
    !git pull

# 2. Sistem Bağımlılıklarını Kur (ZBar)
!apt-get install -y -qq libzbar0

# 3. Python Paketlerini Kur (SABİT VERSİYONLAR)
print("📦 Paketler yükleniyor (requirements.txt baz alınıyor)...")
!pip install -q -r requirements.txt
!pip install -q -e .

# 4. WeChat QR Modellerini Otomatik Kur
if os.path.exists('setup_models.py'):
    print("🧠 Yapay Zeka modelleri kontrol ediliyor...")
    !python3 setup_models.py

# Klasör yapısını hazırla
!mkdir -p inputs/assets
!mkdir -p output

print("✅ Kurulum ve Modeller tamamlandı.")

## 📁 2. order.csv Yükle
İşlenecek QR kod listesini içeren CSV dosyasını yükleyin.

In [ ]:
csv_path = 'inputs/order.csv'

def upload_csv(b):
    clear_output()
    display(HTML("<div style='color: #2980b9; font-weight: bold;'>📁 Lütfen order.csv dosyasını seçin...</div>"))
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        with open(csv_path, 'wb') as f:
            f.write(uploaded[filename])
        display(HTML(f"<div style='color: #27ae60; font-weight: bold;'>✅ {filename} yüklendi.</div>"))
        display(HTML("<p>Şimdi Adım 3'e geçerek assetleri yükleyebilirsiniz.</p>"))

up_csv_btn = widgets.Button(description="📁 order.csv Yükle", button_style='primary', layout={'width': '250px', 'height': '50px'})
up_csv_btn.on_click(upload_csv)
display(up_csv_btn)

## 🖼️ 3. Assetleri Yükle (Logos, GIFs, Backgrounds)
QR kodlarda kullanacağınız görselleri buraya yükleyin.

In [ ]:
def upload_assets(b):
    display(HTML("<div style='color: #2980b9;'>🖼️ Lütfen arka plan/logo dosyalarını seçin (Çoklu seçim yapılabilir)...</div>"))
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest = os.path.join('inputs/assets', filename)
        with open(dest, 'wb') as f:
            f.write(uploaded[filename])
        display(HTML(f"<div style='color: #27ae60;'>✅ {filename} yüklendi.</div>"))

up_asset_btn = widgets.Button(description="🖼️ Asset Yükle", button_style='info', layout={'width': '250px', 'height': '50px'})
up_asset_btn.on_click(upload_assets)
display(up_asset_btn)

## ✍️ 4. Siparişleri Düzenle (Editor)
Tablo üzerinde toplu veya manuel değişiklikler yapın.

In [ ]:
def load_data():
    if not os.path.exists(csv_path): return pd.DataFrame()
    df = pd.read_csv(csv_path)
    cols = ['selected', 'words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name']
    for c in cols:
        if c not in df.columns:
            if c == 'selected': df[c] = True
            elif c == 'version': df[c] = 1
            elif c == 'level': df[c] = 'H'
            elif c in ['contrast', 'brightness']: df[c] = 1.0
            elif c == 'colorized': df[c] = True
            else: df[c] = ""
    return df[cols]

def save_data(df): df.to_csv(csv_path, index=False)

editor_out = widgets.Output()

def refresh_editor():
    with editor_out:
        clear_output()
        df = load_data()
        if df.empty: 
            display(HTML("❌ Tablo bulunamadı. Lütfen Adım 2'yi kullanarak bir CSV yükleyin."))
            return
        
        bulk_btn = widgets.Button(description="🛠️ Toplu Düzenle", button_style='warning', icon='gears')
        bulk_btn.on_click(lambda b: show_bulk_editor())
        refresh_btn = widgets.Button(description="🔄 Yenile", button_style='info', icon='refresh')
        refresh_btn.on_click(lambda b: refresh_editor())
        
        display(widgets.HBox([bulk_btn, refresh_btn]))
        display(HTML("<br>"))

        header = widgets.HBox([
            widgets.Label("Seç", layout={'width': '40px'}),
            widgets.Label("URL / Metin", layout={'width': '200px'}),
            widgets.Label("V", layout={'width': '30px'}),
            widgets.Label("L", layout={'width': '30px'}),
            widgets.Label("Görsel", layout={'width': '120px'}),
            widgets.Label("İşlem", layout={'width': '80px'})
        ])
        display(header)
        
        rows = []
        for i, row in df.iterrows():
            sel = widgets.Checkbox(value=bool(row['selected']), layout={'width': '40px'})
            def on_sel(change, idx=i): 
                d = load_data()
                d.at[idx, 'selected'] = change['new']
                save_data(d)
            sel.observe(on_sel, 'value')
            
            edit = widgets.Button(description="Düzenle", icon='pencil', layout={'width': '80px'}, button_style='primary')
            edit.on_click(lambda b, idx=i: show_row_editor(idx))
            
            r = widgets.HBox([
                sel, 
                widgets.Label(str(row['words'])[:25], layout={'width': '200px'}), 
                widgets.Label(str(row['version']), layout={'width': '30px'}),
                widgets.Label(str(row['level']), layout={'width': '30px'}),
                widgets.Label(str(row['picture']), layout={'width': '120px'}), 
                edit
            ])
            rows.append(r)
        display(widgets.VBox(rows))

def show_row_editor(idx):
    with editor_out:
        clear_output()
        df = load_data()
        row = df.iloc[idx]
        w_words = widgets.Text(value=str(row['words']), description='Words:')
        w_ver = widgets.IntSlider(value=int(row['version']), min=1, max=40, description='Version:')
        w_lvl = widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value=row['level'], description='Level:')
        w_pic = widgets.Text(value=str(row['picture']), description='Picture:')
        w_col = widgets.Checkbox(value=bool(row['colorized']), description='Colorized')
        w_con = widgets.FloatSlider(value=float(row['contrast']), min=0.1, max=3.0, step=0.1, description='Contrast:')
        w_bri = widgets.FloatSlider(value=float(row['brightness']), min=0.1, max=3.0, step=0.1, description='Brightness:')
        w_name = widgets.Text(value=str(row['save_name']), description='Name:')
        
        save = widgets.Button(description="Kaydet", button_style='success', icon='check')
        cancel = widgets.Button(description="İptal", button_style='danger', icon='times')
        
        def on_save(b):
            df.at[idx, 'words'] = w_words.value
            df.at[idx, 'version'] = w_ver.value
            df.at[idx, 'level'] = w_lvl.value
            df.at[idx, 'picture'] = w_pic.value
            df.at[idx, 'colorized'] = w_col.value
            df.at[idx, 'contrast'] = w_con.value
            df.at[idx, 'brightness'] = w_bri.value
            df.at[idx, 'save_name'] = w_name.value
            save_data(df)
            refresh_editor()
            
        save.on_click(on_save)
        cancel.on_click(lambda b: refresh_editor())
        display(widgets.VBox([w_words, w_ver, w_lvl, w_pic, w_col, w_con, w_bri, w_name, widgets.HBox([save, cancel])]))

def show_bulk_editor():
    with editor_out:
        clear_output()
        df = load_data()
        v_check = widgets.Checkbox(description="Versiyon")
        v_val = widgets.IntSlider(value=1, min=1, max=40)
        l_check = widgets.Checkbox(description="Level")
        l_val = widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value='H')
        p_check = widgets.Checkbox(description="Görsel")
        p_val = widgets.Text(placeholder="logo.png")
        apply_b = widgets.Button(description="Uygula", button_style='warning')
        
        def on_apply(b):
            for i in range(len(df)):
                if v_check.value: df.at[i, 'version'] = v_val.value
                if l_check.value: df.at[i, 'level'] = l_val.value
                if p_check.value: df.at[i, 'picture'] = p_val.value
            save_data(df); refresh_editor()
            
        apply_b.on_click(on_apply)
        display(widgets.VBox([widgets.HBox([v_check, v_val]), widgets.HBox([l_check, l_val]), widgets.HBox([p_check, p_val]), apply_b]))

refresh_editor()
display(editor_out)

## 📊 5. İşlem Öncesi Kontrol
Sadece seçili olan satırlar gösterilir.

In [ ]:
df = load_data()
if not df.empty:
    selected_df = df[df['selected'] == True]
    display(HTML("<h3 style='color: #2c3e50;'>🚀 İşleme Alınacak Liste</h3>"))
    display(selected_df)
else:
    print("❌ Liste boş.")

## ⚙️ 6. Üretimi Başlat (Process)
QR kod üretimini başlatır.

In [ ]:
from batch_process import process_items
output_dir = 'output'
if os.path.exists(output_dir): shutil.rmtree(output_dir)
os.makedirs(output_dir)

df = load_data()
selected_data = df[df['selected'] == True].to_dict('records')
process_reports = []

print(f"🚀 {len(selected_data)} adet QR kod üretiliyor...")
for msg, path, report in process_items(selected_data, 'inputs/assets', output_dir, auto_repair=False):
    if msg: print(msg)
    if report: process_reports.append(report)

print("\n✅ Üretim tamamlandı.")

## ✨ 7. Sonuçları Önizle
Üretilen tüm görselleri burada görebilirsiniz.

In [ ]:
import base64
if os.path.exists(output_dir):
    files_list = sorted([f for f in os.listdir(output_dir) if f.lower().endswith(('.png', '.gif', '.jpg'))])
    html = '<div style="display: flex; flex-wrap: wrap; gap: 10px;">'
    for f in files_list:
        with open(os.path.join(output_dir, f), "rb") as img:
            enc = base64.b64encode(img.read()).decode('utf-8')
        mime = "image/gif" if f.endswith('.gif') else "image/png"
        html += f'<div style="text-align:center; border:1px solid #ddd; padding:5px;"><img src="data:{mime};base64,{enc}" width="150"><br><small>{f}</small></div>'
    html += '</div>'
    display(HTML(html))

## 🔍 8. Kalite Kontrol (QC)
QR kodların okunabilirliği otomatik olarak test edilir (Gelişmiş pyzbar motoru aktif).

In [ ]:
qc_html = "<table border='1' style='border-collapse: collapse; width: 100%; text-align: center;'>"
qc_html += "<tr style='background: #2c3e50; color: white;'><th>Dosya</th><th>Durum</th><th>İçerik</th></tr>"
for r in process_reports:
    status = r.get('scannable', 'Unknown')
    color = "#27ae60" if "✅" in status else "#e74c3c"
    qc_html += f"<tr><td>{r['output_file']}</td><td style='color: {color}; font-weight: bold;'>{status}</td><td>{r.get('scanned_data', '-')}</td></tr>"
qc_html += "</table>"
display(HTML(qc_html))

## 🛠️ 9. Akıllı Onarım & İndir
Hatalı kodları onarır ve sonuçları ZIP olarak paketler.

In [ ]:
failed = [r for r in process_reports if "❌" in r.get('scannable', '')]

def run_repair(b):
    global process_reports
    clear_output()
    display(HTML("⏳ Hatalı kodlar onarılıyor (High Contrast Mode)... "))
    to_repair = [r for r in process_reports if "❌" in r.get('scannable', '')]
    new_reports = []
    for msg, path, report in process_items(to_repair, 'inputs/assets', 'output', auto_repair=True):
        if msg: print(msg)
        if report: new_reports.append(report)
    for nr in new_reports:
        for i, orp in enumerate(process_reports):
            if orp['output_file'] == nr['output_file']: process_reports[i] = nr
    display(HTML("✅ Onarım tamamlandı! Lütfen Adım 8'i tekrar çalıştırın veya aşağıdan indirin."))

def download_results(b):
    import time
    zip_name = f"amazing_qr_results_{int(time.time())}.zip"
    !zip -rj {zip_name} output/
    files.download(zip_name)

if failed:
    display(HTML(f"<h4 style='color: #e67e22;'>⚠️ {len(failed)} adet QR okunamadı.</h4>"))
    repair_btn = widgets.Button(description="🛠️ Onarımı Başlat", button_style='warning')
    repair_btn.on_click(run_repair)
    display(repair_btn)
else:
    display(HTML("<h4 style='color: #27ae60;'>✅ Tüm QR kodlar başarılı!</h4>"))

dl_btn = widgets.Button(description="📥 Sonuçları ZIP İndir", button_style='success')
dl_btn.on_click(download_results)
display(dl_btn)